In [128]:
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

In [129]:
# 사이킷런을 이용해 당뇨병 데이터 다운로드
diabetes = fetch_openml(
    data_id=37,
    as_frame=True,
    parser="pandas"
)

# 특성 데이터 복사
df = diabetes.data.copy()

# 정답 추가 (0 = 음성, 1 = 양성)
df["target"] = diabetes.target.map({
    "tested_negative": 0,
    "tested_positive": 1
}).astype(int)

print(df.shape)
print(df.head(3))

(768, 9)
   preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1


### 1. 데이터 탐색

판다스를 이용하여 데이터를 탐색하세요.

In [130]:
print(f"행 개수 : {df.shape[0]}")
print(f"열 개수 : {df.shape[1]}")
print(f"상위 5개 행 : {df.head(5)}")
print(df.info())
print(df.describe())
print(df["target"].value_counts())
print(f"양성 음성 비율 : {(df["target"].sum()/len(df["target"]))*100}")

행 개수 : 768
열 개수 : 9
상위 5개 행 :    preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1
3     1    89    66    23    94  28.1  0.167   21       0
4     0   137    40    35   168  43.1  2.288   33       1
<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preg    768 non-null    int64  
 1   plas    768 non-null    int64  
 2   pres    768 non-null    int64  
 3   skin    768 non-null    int64  
 4   insu    768 non-null    int64  
 5   mass    768 non-null    float64
 6   pedi    768 non-null    float64
 7   age     768 non-null    int64  
 8   target  768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None
             preg        plas        pres        skin        insu 

1. 양성은 34.89%
2. 가장 큰 특성 : insu, 가장 작은 특성 : pedi
3. 값이 큰 특성쪽으로 범위가 치우쳐지게 된다.

### 2. 결측치 탐색

데이터에서 누락된 측정값을 찾아보세요.

In [131]:
print(df.isna().sum())

preg      0
plas      0
pres      0
skin      0
insu      0
mass      0
pedi      0
age       0
target    0
dtype: int64


In [132]:
print(df["plas"].loc[df["plas"]==0].value_counts())
print(df["pres"].loc[df["pres"]==0].value_counts())
print(df["skin"].loc[df["skin"]==0].value_counts())
print(df["insu"].loc[df["insu"]==0].value_counts())
print(df["mass"].loc[df["mass"]==0].value_counts())

plas
0    5
Name: count, dtype: int64
pres
0    35
Name: count, dtype: int64
skin
0    227
Name: count, dtype: int64
insu
0    374
Name: count, dtype: int64
mass
0.0    11
Name: count, dtype: int64


In [133]:
import numpy as np
df[["plas","pres","skin","insu","mass"]]=df[["plas","pres","skin","insu","mass"]].replace(0,np.nan)
print(df.isna().sum())
mask = df.isna().any(axis = 1)
print(df[mask])

preg        0
plas        5
pres       35
skin      227
insu      374
mass       11
pedi        0
age         0
target      0
dtype: int64
     preg   plas  pres  skin  insu  mass   pedi  age  target
0       6  148.0  72.0  35.0   NaN  33.6  0.627   50       1
1       1   85.0  66.0  29.0   NaN  26.6  0.351   31       0
2       8  183.0  64.0   NaN   NaN  23.3  0.672   32       1
5       5  116.0  74.0   NaN   NaN  25.6  0.201   30       0
7      10  115.0   NaN   NaN   NaN  35.3  0.134   29       0
..    ...    ...   ...   ...   ...   ...    ...  ...     ...
761     9  170.0  74.0  31.0   NaN  44.0  0.403   43       1
762     9   89.0  62.0   NaN   NaN  22.5  0.142   33       0
764     2  122.0  70.0  27.0   NaN  36.8  0.340   27       0
766     1  126.0  60.0   NaN   NaN  30.1  0.349   47       1
767     1   93.0  70.0  31.0   NaN  30.4  0.315   23       0

[376 rows x 9 columns]


1. 값이 nan이 아닌 0이었기에 발견되지 않았다
2. 0이 결측치가 아닌 열도 같이 변경되기 때문에 데이터가 오염된다

### 3. 특성과 정답 분리 및 데이터 분할

머신러닝을 학습하기 위한 데이터를 준비하세요.

In [134]:
x = df.drop(columns="target")
y = df["target"]
print(x.shape)
print(y.shape)
train_x,test_x,train_y,test_y = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
print(train_x.shape,train_y.shape)
print(test_x.shape,test_y.shape)

(768, 8)
(768,)
(614, 8) (614,)
(154, 8) (154,)


1. 한 데이터로만 학습하고 테스트하면 이미 학습된 문제를 풀기때문에 정확한 채점을 할 수 없어서 데이터 안에서 학습 데이터와 테스트 데이터로 나눈다.
2. 분류 할 데이터를 평등하게 학습시키기 위해서  y값으로 공정하게 나누라는 표시이다.
3. 항상 같은 값으로 테스트가 가능하다.

### 4. 결측치 처리

3번에서 분리한 학습 데이터와 테스트 데이터의 결측치를 처리하세요.

In [135]:
train_xc = train_x.fillna(train_x.median())
test_xc = test_x.fillna(train_x.median())
print(train_xc.isna().sum())
print(test_xc.isna().sum())
print("처리 전:",train_x.shape,"처리 후:",train_xc.shape)
print("처리 전:",test_x.shape,"처리 후:",test_xc.shape)

preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64
preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64
처리 전: (614, 8) 처리 후: (614, 8)
처리 전: (154, 8) 처리 후: (154, 8)


1. 평균보다 이상치의 값에 덜 영향받아서 데이터가 크게 오염되지 않는다
2. 테스트데이터와 학습데이터의 중앙값은 다를 수 있기에 학습데이터로 학습했을때와 중앙값이 다르면 테스트가 크게 틀릴 수 있다

### 5. 첫 번째 KNN 학습

결측치 처리가 완료된 데이터를 이용하여 KNN을 학습하세요.

In [136]:
model_k = KNeighborsClassifier(n_neighbors=5)
model_k.fit(train_xc,train_y)
print("학습 정확도",model_k.score(train_xc,train_y))
print("테스트 정확도",model_k.score(test_xc,test_y))
pred = model_k.predict(test_xc)
for i in range(10):
    print(f"{i+1} : 예측 결과 : {pred[i]}, 실제 답 : {test_y.iloc[i]}")

학습 정확도 0.8013029315960912
테스트 정확도 0.6753246753246753
1 : 예측 결과 : 1, 실제 답 : 0
2 : 예측 결과 : 0, 실제 답 : 0
3 : 예측 결과 : 0, 실제 답 : 0
4 : 예측 결과 : 0, 실제 답 : 1
5 : 예측 결과 : 0, 실제 답 : 0
6 : 예측 결과 : 0, 실제 답 : 0
7 : 예측 결과 : 0, 실제 답 : 1
8 : 예측 결과 : 1, 실제 답 : 1
9 : 예측 결과 : 0, 실제 답 : 0
10 : 예측 결과 : 1, 실제 답 : 0


1. 주변 몇개까지의 값을 보는지
2. 학습정확도는 학습이 얼마나 잘 되었는지 테스트 정확도는 실제로 얼마나 잘 학습했는지
3. 학습정확도는 이미 본 문제에서 하고 테스트 정확도는 처음보는 문제들을 풀기에 전반적으로 테스트 정확도가 낮다

### 6. KNN 성능 평가

5번에서 학습한 KNN의 예측 결과를 평가하세요.

다음 항목을 출력하세요.

In [137]:
print("혼동행렬 : \n",confusion_matrix(test_y,pred))
print("accuracy_score : " ,accuracy_score(test_y,pred))
print("precision_score : ",precision_score(test_y,pred)  )
print("recall_score : ",recall_score(test_y,pred)  )
print("f1_score : ",f1_score(test_y,pred)  )
print(classification_report(test_y,pred)  )



혼동행렬 : 
 [[77 23]
 [27 27]]
accuracy_score :  0.6753246753246753
precision_score :  0.54
recall_score :  0.5
f1_score :  0.5192307692307693
              precision    recall  f1-score   support

           0       0.74      0.77      0.75       100
           1       0.54      0.50      0.52        54

    accuracy                           0.68       154
   macro avg       0.64      0.64      0.64       154
weighted avg       0.67      0.68      0.67       154



1. 27
2. 23
3. 정확도는 TP + TN / 전체 재현율은 TP /TP+ FN이므로 TP가 낮고 FN,TN이 높다면 발생할 수 있다.
4. 재현율을 높게 봐서 최대한 양성을 놓치는 일이 없게 중요하게 봐야한다.

### 7. k값에 따른 성능 비교

KNN의 k값을 변경하면서 학습 정확도와 테스트 정확도가 어떻게 변하는지 확인하세요.

In [138]:
k_s =[1, 3, 5, 7, 10, 15, 20, 30]
for i in k_s:
    model_k = KNeighborsClassifier(n_neighbors=i)
    model_k.fit(train_xc,train_y)
    train_score = model_k.score(train_xc,train_y)
    test_score = model_k.score(test_xc,test_y)
    pred = model_k.predict(test_xc)
    reall = recall_score(test_y,pred)
    print(f"k : {i}, train_acc : {train_score}, test_acc : {test_score}")

k : 1, train_acc : 1.0, test_acc : 0.6818181818181818
k : 3, train_acc : 0.8420195439739414, test_acc : 0.6818181818181818
k : 5, train_acc : 0.8013029315960912, test_acc : 0.6753246753246753
k : 7, train_acc : 0.7850162866449512, test_acc : 0.6753246753246753
k : 10, train_acc : 0.7980456026058632, test_acc : 0.6688311688311688
k : 15, train_acc : 0.7915309446254072, test_acc : 0.6558441558441559


k : 20, train_acc : 0.7882736156351792, test_acc : 0.7207792207792207
k : 30, train_acc : 0.7654723127035831, test_acc : 0.7207792207792207


In [139]:
# k : 1, train_acc : 1.0, test_acc : 0.6818181818181818
# k : 3, train_acc : 0.8420195439739414, test_acc : 0.6818181818181818
# k : 5, train_acc : 0.8013029315960912, test_acc : 0.6753246753246753
# k : 7, train_acc : 0.7850162866449512, test_acc : 0.6753246753246753
# k : 10, train_acc : 0.7980456026058632, test_acc : 0.6688311688311688
# k : 15, train_acc : 0.7915309446254072, test_acc : 0.6558441558441559
# k : 20, train_acc : 0.7882736156351792, test_acc : 0.7207792207792207
# k : 30, train_acc : 0.7654723127035831, test_acc : 0.7207792207792207

1. 1
2. 20, 30
3. 주변의 특성이 적기에 학습량이 적어서 쉽게 맞춘다
4. 너무나 많은 특성을 한번에 보기에 제대로 된 학습이 이루이지지 않을 수 있다.
5. 학습정확도 말고도 많은 지표가 있기에 상황에 맞는 지표를 선택해서 봐야한다.

### 8. StandardScaler 적용

이번에는 `StandardScaler`로 특성을 표준화한 뒤 동일한 KNN을 학습하세요.

In [140]:
scaler = StandardScaler()
scaler.fit(train_xc)
train_xcs = scaler.transform(train_xc)
test_xcs = scaler.transform(test_xc)
train_xcs_df = pd.DataFrame(train_xcs,columns=train_xc.columns)
print(train_xcs_df.describe())

               preg          plas          pres          skin          insu  \
count  6.140000e+02  6.140000e+02  6.140000e+02  6.140000e+02  6.140000e+02   
mean  -6.943414e-17 -1.099374e-16  3.095606e-16 -3.471707e-17 -4.339634e-18   
std    1.000815e+00  1.000815e+00  1.000815e+00  1.000815e+00  1.000815e+00   
min   -1.153338e+00 -2.190541e+00 -3.924957e+00 -2.480958e+00 -1.559139e+00   
25%   -8.513551e-01 -7.562208e-01 -6.636760e-01 -4.549828e-01 -2.249692e-01   
50%   -2.473897e-01 -1.558076e-01 -1.141979e-02 -4.766137e-03 -1.614373e-01   
75%    6.585583e-01  6.113872e-01  6.408364e-01  3.328964e-01 -9.790537e-02   
max    3.980368e+00  2.579408e+00  4.065181e+00  7.874025e+00  7.703814e+00   

               mass          pedi           age  
count  6.140000e+02  6.140000e+02  6.140000e+02  
mean   1.649061e-16 -1.099374e-16 -1.084908e-16  
std    1.000815e+00  1.000815e+00  1.000815e+00  
min   -2.089621e+00 -1.192094e+00 -1.045895e+00  
25%   -7.073644e-01 -7.042617e-01 -7.9

In [141]:
model_k2  = KNeighborsClassifier(n_neighbors=5)
model_k2.fit(train_xcs,train_y)
print(f"trian_acc : {model_k2.score(train_xcs,train_y)}\ntest_acc : {model_k2.score(test_xcs,test_y)}")

print(f"recall_score : {recall_score(test_y,model_k2.predict(test_xcs))}")

trian_acc : 0.8289902280130294
test_acc : 0.7532467532467533
recall_score : 0.6111111111111112


1.                preg          plas          pres          skin          insu  \
  
mean  -6.943414e-17 -1.099374e-16  3.095606e-16 -3.471707e-17 -4.339634e-18   
std    1.000815e+00  1.000815e+00  1.000815e+00  1.000815e+00  1.000815e+00   
 

               mass          pedi           age  
 
mean   1.649061e-16 -1.099374e-16 -1.084908e-16  
std    1.000815e+00  1.000815e+00  1.000815e+00  
2. 학습정확도가 2% 테스트 정확도가 8% 상승했다
3. 큰 편차를 지닌 특성과 작은 편차를 지닌 특성의 값들이 조정되면서 모델이 전체적으로 특성을 더 잘 보게 되었다


### 9. MinMaxScaler 적용 및 비교

이번에는 `MinMaxScaler`를 적용하고 KNN의 성능을 확인하세요.

In [142]:
minmax = MinMaxScaler()
minmax.fit(train_xc)
train_xcm = minmax.transform(train_xc)
test_xcm = minmax.transform(test_xc)
train_xcm_df = pd.DataFrame(train_xcm,columns=x.columns)
print(train_xcm_df.describe())

             preg        plas        pres        skin        insu        mass  \
count  614.000000  614.000000  614.000000  614.000000  614.000000  614.000000   
mean     0.224660    0.459238    0.491225    0.239591    0.168320    0.291374   
std      0.194950    0.209817    0.125256    0.096651    0.108045    0.139553   
min      0.000000    0.000000    0.000000    0.000000    0.000000    0.000000   
25%      0.058824    0.300699    0.408163    0.195652    0.144033    0.192740   
50%      0.176471    0.426573    0.489796    0.239130    0.150892    0.290389   
75%      0.352941    0.587413    0.571429    0.271739    0.157750    0.374233   
max      1.000000    1.000000    1.000000    1.000000    1.000000    1.000000   

             pedi         age  
count  614.000000  614.000000  
mean     0.175246    0.206107  
std      0.147127    0.197224  
min      0.000000    0.000000  
25%      0.071715    0.050000  
50%      0.132962    0.133333  
75%      0.247327    0.333333  
max      1.000

In [143]:
model_k3 = KNeighborsClassifier(n_neighbors=5)
model_k3.fit(train_xcm,train_y)
print(f"trian_acc : {model_k3.score(train_xcm,train_y)}\ntest_acc : {model_k3.score(test_xcm,test_y)}")

print(f"recall_score : {recall_score(test_y,model_k3.predict(test_xcm))}")
pred = model_k3.predict(test_xcm)
print(classification_report(test_y,pred))

trian_acc : 0.8322475570032574
test_acc : 0.7337662337662337
recall_score : 0.5555555555555556
              precision    recall  f1-score   support

           0       0.78      0.83      0.80       100
           1       0.64      0.56      0.59        54

    accuracy                           0.73       154
   macro avg       0.71      0.69      0.70       154
weighted avg       0.73      0.73      0.73       154



| KNN 전처리 방법 | 학습 정확도 | 테스트 정확도 | 재현율 |
|:---:|:---:|:---:|:---:|
| 스케일링 없음 | 0.8013029315960912 | 0.6753246753246753 | 0.5 |
| StandardScaler | 0.8289902280130294 | 0.7532467532467533 | 0.6111111111111112 |
| MinMaxScaler | 0.8322475570032574 | 0.7337662337662337 | 0.5555555555555556 |

1.  StandardScaler
2.  StandardScaler
3.  두개의 공식이 다르니 정확도의 합이 높아지는 방법이 있을 수 있고 재현율이 높아지는 방법이 있을 수 있다
4.  데이터 자체가 이미 크게 벗어나지 않아서 스케일링이 효과가 없을 수 있다.

### 10. 종합 — KNN 설정 선택 및 최종 분석

지금까지 실험한 결과를 이용하여 가장 적절하다고 생각하는 KNN 설정을 선택하세요.

**1단계 — 스케일링 방법 선택**

다음 중 하나를 선택하세요.

In [144]:
for i in k_s:
    model_k2 = KNeighborsClassifier(n_neighbors=i)
    model_k2.fit(train_xcs,train_y)
    train_score = model_k2.score(train_xcs,train_y)
    test_score = model_k2.score(test_xcs,test_y)
    pred = model_k2.predict(test_xcs)
    recall = recall_score(test_y,pred)
    print(f"k : {i}, train_acc : {train_score}, test_acc : {test_score}, recall_score : {recall}" )

k : 1, train_acc : 1.0, test_acc : 0.7207792207792207, recall_score : 0.5555555555555556
k : 3, train_acc : 0.8469055374592834, test_acc : 0.7207792207792207, recall_score : 0.6111111111111112
k : 5, train_acc : 0.8289902280130294, test_acc : 0.7532467532467533, recall_score : 0.6111111111111112
k : 7, train_acc : 0.8175895765472313, test_acc : 0.7272727272727273, recall_score : 0.5555555555555556
k : 10, train_acc : 0.8045602605863192, test_acc : 0.7402597402597403, recall_score : 0.5370370370370371
k : 15, train_acc : 0.8029315960912052, test_acc : 0.7402597402597403, recall_score : 0.5555555555555556
k : 20, train_acc : 0.7882736156351792, test_acc : 0.7597402597402597, recall_score : 0.5555555555555556
k : 30, train_acc : 0.7882736156351792, test_acc : 0.7402597402597403, recall_score : 0.5185185185185185


StandardScaler
| k | 학습 정확도 | 테스트 정확도 | 재현율 |
|:---:|:---:|:---:|:---:|
| 1 | 1.0 | 0.7207792207792207 | 0.5555555555555556 |
| 3 | 0.8469055374592834 | 0.7207792207792207 | 0.6111111111111112 |
| 5 | 0.8289902280130294 | 0.7532467532467533 | 0.6111111111111112 |
| 7 | 0.8175895765472313 | 0.7272727272727273 | 0.5555555555555556 |
| 10 | 0.8045602605863192 | 0.7402597402597403 | 0.5370370370370371 |
| 15 | 0.8029315960912052 | 0.7402597402597403 | 0.5555555555555556 |
| 20 | 0.7882736156351792 | 0.7597402597402597 | 0.5555555555555556 |
| 30 | 0.7882736156351792 | 0.7402597402597403 | 0.5185185185185185 |

1. StandardScaler
2. 5

In [145]:
model_F = KNeighborsClassifier(n_neighbors=5)
model_F.fit(train_xcs,train_y)
train_score = model_F.score(train_xcs,train_y)
test_score = model_F.score(test_xcs,test_y)
pred = model_F.predict(test_xcs)
print(f"k : {i}, train_acc : {train_score}, test_acc : {test_score}" )
print("혼동행렬 : \n",confusion_matrix(test_y,pred))
print("accuracy_score : " ,accuracy_score(test_y,pred))
print("precision_score : ",precision_score(test_y,pred)  )
print("recall_score : ",recall_score(test_y,pred)  )
print("f1_score : ",f1_score(test_y,pred)  )
print(classification_report(test_y,pred)  )


k : 30, train_acc : 0.8289902280130294, test_acc : 0.7532467532467533
혼동행렬 : 
 [[83 17]
 [21 33]]
accuracy_score :  0.7532467532467533
precision_score :  0.66
recall_score :  0.6111111111111112
f1_score :  0.6346153846153846
              precision    recall  f1-score   support

           0       0.80      0.83      0.81       100
           1       0.66      0.61      0.63        54

    accuracy                           0.75       154
   macro avg       0.73      0.72      0.72       154
weighted avg       0.75      0.75      0.75       154



1. StandardScaler, 5
2. 이 설정보다 양성클래스에서 재현율이 높은 설정이 없어서
3. 전체적인 정확도, 재현율, 정밀도등의 세부지표가 전부 상승
4. 5에서 작아질 수록 학습정확도가 높아지고 나머지가 낮아짐 5에서 멀어질 수록 점점 학습정확도가 떨어지고 재현율도 합께 떨어짐
5. 재현율을 더 중요하게 고려함, 양성을 찾는것이 목적이기에 오탐보다는 놓치는게 없는것을 더 중요시함
6. 모델이 학습데이터를 기준으로 학습하기에 이와 경향이 달라질 수 있는 테스트 데이터의 값을 이용하면 학습이 무용이 된다고 생각합니다.
7. 전처리 그냥 전처리 단계에서 어떻게 스케일링 해야하는가 어떻게 나눌것인가 어떤 하이퍼 파라미터를 쓸 것인가등을 선택하는 학습 전 모든 단계가 중요하다고 생각합니다.